In [0]:
%pip install langchain-databricks langchain-core tiktoken

In [0]:
dbutils.library.restartPython()

In [0]:

import json
import time
import statistics
import textwrap
import random
import numpy as np
import tiktoken
from langchain_databricks import ChatDatabricks
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage

SEED = 42
MAX_ITERATIONS = 40
NUM_RUNS = 3
SEEDS = [42, 123, 456]
MODEL_ENDPOINT = "databricks-claude-haiku-4-5"

_enc = tiktoken.get_encoding("cl100k_base")
def count_tokens(text: str) -> int:
    return len(_enc.encode(str(text)))

# Generate fixed input data
_rng = random.Random(SEED)
_names = ["Alice","Bob","Charlie","Diana","Eve","Frank","Grace","Hank",
          "Ivy","Jack","Karen","Leo","Mona","Nick","Olivia","Pat",
          "Quinn","Rose","Sam","Tina","Uma","Vic","Wendy","Xander","Yuki","Zane"]
_cities = ["Toronto","Vancouver","Montreal","Calgary","Ottawa","Edmonton",
           "Winnipeg","Halifax","Victoria","Quebec City"]
_depts = ["engineering","sales","marketing","support","hr","finance",
          "legal","ops","research","design"]
_all_tags = ["python","java","go","rust","sql","ml","devops","frontend","backend","mobile"]

RECORDS = []
for _i in range(10000):
    RECORDS.append({
        "id": _i, "name": _rng.choice(_names), "city": _rng.choice(_cities),
        "department": _rng.choice(_depts),
        "salary": round(_rng.uniform(40000, 180000), 2),
        "years": _rng.randint(0, 30),
        "rating": round(_rng.uniform(1.0, 5.0), 1),
        "active": _rng.random() > 0.15,
        "projects": _rng.randint(0, 50),
        "tags": _rng.sample(_all_tags, k=_rng.randint(1, 5)),
    })

DEFAULT_CODE = textwrap.dedent('''\
def process_records(records):
    """Process employee records: filter, transform, aggregate, report."""

    # Step 1: Filter active employees with 3+ years
    filtered = []
    for r in records:
        if r["active"] == True and r["years"] >= 3:
            filtered.append(r)

    # Step 2: Group by department
    departments = {}
    for r in filtered:
        dept = r["department"]
        if dept not in departments:
            departments[dept] = []
        departments[dept].append(r)

    # Step 3: Per-department stats
    report = {}
    for dept in departments:
        employees = departments[dept]
        total_salary = 0
        total_rating = 0
        total_projects = 0
        senior_count = 0
        all_tags = []
        names_list = ""

        for emp in employees:
            total_salary = total_salary + emp["salary"]
            total_rating = total_rating + emp["rating"]
            total_projects = total_projects + emp["projects"]
            if emp["years"] >= 10:
                senior_count = senior_count + 1
            for tag in emp["tags"]:
                all_tags.append(tag)
            names_list = names_list + emp["name"] + ", "

        count = len(employees)
        avg_salary = total_salary / count if count > 0 else 0
        avg_rating = total_rating / count if count > 0 else 0

        # Count tag frequencies
        tag_counts = {}
        for tag in all_tags:
            if tag in tag_counts:
                tag_counts[tag] = tag_counts[tag] + 1
            else:
                tag_counts[tag] = 1

        # Sort tags by frequency (bubble sort)
        sorted_tags = []
        for tag in tag_counts:
            sorted_tags.append((tag, tag_counts[tag]))
        for i in range(len(sorted_tags)):
            for j in range(i + 1, len(sorted_tags)):
                if sorted_tags[j][1] > sorted_tags[i][1]:
                    temp = sorted_tags[i]
                    sorted_tags[i] = sorted_tags[j]
                    sorted_tags[j] = temp

        top_tags = []
        for k in range(min(3, len(sorted_tags))):
            top_tags.append(sorted_tags[k][0])

        report[dept] = {
            "count": count,
            "avg_salary": round(avg_salary, 2),
            "avg_rating": round(avg_rating, 1),
            "total_projects": total_projects,
            "senior_count": senior_count,
            "senior_pct": round(senior_count / count * 100, 1) if count > 0 else 0,
            "top_tags": top_tags,
            "names": names_list.rstrip(", "),
        }

    # Step 4: Sort departments by avg_salary descending (bubble sort)
    sorted_depts = []
    for dept in report:
        sorted_depts.append((dept, report[dept]))
    for i in range(len(sorted_depts)):
        for j in range(i + 1, len(sorted_depts)):
            if sorted_depts[j][1]["avg_salary"] > sorted_depts[i][1]["avg_salary"]:
                temp = sorted_depts[i]
                sorted_depts[i] = sorted_depts[j]
                sorted_depts[j] = temp

    # Step 5: Build summary string
    summary = ""
    for dept, stats in sorted_depts:
        summary = summary + dept.upper() + "\\n"
        summary = summary + "  Headcount: " + str(stats["count"]) + "\\n"
        summary = summary + "  Avg Salary: $" + str(stats["avg_salary"]) + "\\n"
        summary = summary + "  Avg Rating: " + str(stats["avg_rating"]) + "\\n"
        summary = summary + "  Projects: " + str(stats["total_projects"]) + "\\n"
        summary = summary + "  Senior: " + str(stats["senior_pct"]) + "%\\n"
        summary = summary + "  Top Skills: " + ", ".join(stats["top_tags"]) + "\\n"
        summary = summary + "\\n"

    return summary, report
''')

def benchmark_code(code_str: str, n_runs: int = 20) -> dict:
    """Execute code and benchmark it."""
    namespace = {}
    try:
        exec(code_str, namespace)
    except Exception as e:
        return {"error": f"Compilation error: {e}", "time_ms": float("inf"),
                "speedup": 0.0, "correct": False}

    if "process_records" not in namespace:
        return {"error": "Function 'process_records' not found", "time_ms": float("inf"),
                "speedup": 0.0, "correct": False}

    func = namespace["process_records"]

    try:
        summary, report = func(RECORDS)
        if not isinstance(report, dict) or not isinstance(summary, str):
            return {"error": "Wrong return type", "time_ms": float("inf"),
                    "speedup": 0.0, "correct": False}
        if len(report) == 0:
            return {"error": "Empty report", "time_ms": float("inf"),
                    "speedup": 0.0, "correct": False}
        for dept_data in report.values():
            required = {"count","avg_salary","avg_rating","total_projects",
                        "senior_count","senior_pct","top_tags","names"}
            if not required.issubset(dept_data.keys()):
                return {"error": f"Missing keys: {required - set(dept_data.keys())}",
                        "time_ms": float("inf"), "speedup": 0.0, "correct": False}
    except Exception as e:
        return {"error": f"Runtime error: {e}", "time_ms": float("inf"),
                "speedup": 0.0, "correct": False}

    times = []
    for _ in range(n_runs):
        t0 = time.perf_counter()
        func(RECORDS)
        t1 = time.perf_counter()
        times.append((t1 - t0) * 1000)

    median_ms = statistics.median(times)
    return {
        "time_ms": round(median_ms, 3),
        "std_ms": round(statistics.stdev(times), 3) if len(times) > 1 else 0.0,
        "min_ms": round(min(times), 3),
        "max_ms": round(max(times), 3),
        "correct": True,
        "n_departments": len(report),
        "summary_length": len(summary),
        "code_lines": len(code_str.strip().split("\n")),
    }

_baseline_result = benchmark_code(DEFAULT_CODE)
BASELINE_TIME_MS = _baseline_result["time_ms"]
print(f"Baseline: {BASELINE_TIME_MS:.3f} ms")

def format_benchmark_result(result: dict, code_str: str) -> str:
    """Format benchmark result for LLM consumption."""
    lines = []
    if "error" in result and result.get("time_ms") == float("inf"):
        lines.append(f"ERROR: {result['error']}")
        lines.append("")
        lines.append("Code submitted:")
        lines.append(code_str)
        return "\n".join(lines)

    speedup = BASELINE_TIME_MS / result["time_ms"] if result["time_ms"] > 0 else 0
    lines.append(f"Execution time: {result['time_ms']:.3f} ms (median of 20 runs)")
    lines.append(f"Baseline time:  {BASELINE_TIME_MS:.3f} ms")
    lines.append(f"Speedup:        {speedup:.2f}x")
    lines.append(f"Std dev:        {result['std_ms']:.3f} ms")
    lines.append(f"Min/Max:        {result['min_ms']:.3f} / {result['max_ms']:.3f} ms")
    lines.append(f"Correct output: {result['correct']}")
    lines.append(f"Code lines:     {result['code_lines']}")
    lines.append("")
    lines.append("Current code:")
    lines.append(code_str)
    return "\n".join(lines)

OPTIMIZER_SYSTEM_PROMPT = """You are a Python performance optimization agent.

You are optimizing a data processing function `process_records(records)` that:
1. Filters active employees with 3+ years experience
2. Groups by department
3. Computes per-department statistics (avg salary, avg rating, projects, senior %, top tags)
4. Sorts departments by avg salary descending
5. Builds a summary string and returns (summary, report)

The function must:
- Accept a list of dicts with keys: id, name, city, department, salary, years, rating, active, projects, tags
- Return a tuple (summary_string, report_dict)
- The report dict must have the same keys per department: count, avg_salary, avg_rating, total_projects, senior_count, senior_pct, top_tags, names
- Produce CORRECT results (same logic, faster execution)

Optimization strategies to consider:
- Use list comprehensions and generator expressions
- Use collections.Counter, collections.defaultdict
- Use str.join() instead of string concatenation
- Use sorted() with key functions instead of bubble sort
- Reduce redundant iterations
- Use f-strings for formatting

Rules:
1. The function signature must remain: def process_records(records)
2. The output must be functionally equivalent
3. You may import standard library modules (collections, itertools, operator, etc.)
4. Do NOT use numpy, pandas, or any third-party libraries
5. Focus on algorithmic and Pythonic improvements

Return your response as:
STRATEGY: <one sentence about what you're optimizing>
CODE: <your complete function code, everything between CODE: and the end>"""

In [0]:

def run_stateless(run_seed: int = SEED) -> dict:
    """Stateless code optimization. Full history reconstructed each iteration."""
    optimizer = ChatDatabricks(endpoint=MODEL_ENDPOINT, max_tokens=2048)

    baseline = benchmark_code(DEFAULT_CODE)
    history = [{
        "iteration": 0, "code": DEFAULT_CODE,
        "time_ms": baseline["time_ms"], "speedup": 1.0,
        "correct": baseline["correct"],
        "result_text": format_benchmark_result(baseline, DEFAULT_CODE),
    }]
    best_time = baseline["time_ms"]
    best_code = DEFAULT_CODE

    total_input_tokens = 0
    total_output_tokens = 0
    per_iter_input_tokens = []

    start_time = time.time()

    for iteration in range(1, MAX_ITERATIONS + 1):
        # Reconstruct FULL context each time (stateless)
        history_text = "Full optimization history:\n\n"
        for h in history:
            speedup = BASELINE_TIME_MS / h["time_ms"] if h["time_ms"] > 0 and h["time_ms"] != float("inf") else 0
            history_text += f"=== Iteration {h['iteration']} (time={h['time_ms']:.3f}ms, speedup={speedup:.2f}x, correct={h['correct']}) ===\n"
            history_text += f"Code:\n{h['code']}\n\n"
            history_text += f"Benchmark:\n{h['result_text']}\n\n"

        history_text += f"\nBaseline time: {BASELINE_TIME_MS:.3f} ms\n"
        history_text += f"Current best: {best_time:.3f} ms ({BASELINE_TIME_MS/best_time:.2f}x)\n"
        history_text += f"\nPropose the next optimized version of process_records()."

        inp_tok = count_tokens(OPTIMIZER_SYSTEM_PROMPT + history_text)
        total_input_tokens += inp_tok
        per_iter_input_tokens.append(inp_tok)

        response = optimizer.invoke([
            SystemMessage(content=OPTIMIZER_SYSTEM_PROMPT),
            HumanMessage(content=history_text),
        ])
        total_output_tokens += count_tokens(response.content)

        try:
            new_code = _extract_code(response.content)
        except Exception:
            new_code = DEFAULT_CODE

        result = benchmark_code(new_code)
        speedup = BASELINE_TIME_MS / result["time_ms"] if result["time_ms"] > 0 and result["time_ms"] != float("inf") else 0

        history.append({
            "iteration": iteration, "code": new_code,
            "time_ms": result["time_ms"], "speedup": speedup,
            "correct": result.get("correct", False),
            "result_text": format_benchmark_result(result, new_code),
        })

        if result.get("correct", False) and result["time_ms"] < best_time:
            best_time = result["time_ms"]
            best_code = new_code

        print(f"  Iter {iteration}: {result['time_ms']:.3f}ms ({speedup:.2f}x) "
              f"correct={result.get('correct',False)} (best={best_time:.3f}ms, inp_tok={inp_tok})")

    elapsed = time.time() - start_time

    return {
        "method": "stateless", "task": "code_opt", "run_seed": run_seed,
        "history": [{"iteration": h["iteration"], "time_ms": h["time_ms"],
                      "speedup": h["speedup"], "correct": h["correct"]} for h in history],
        "best_time_ms": best_time,
        "best_speedup": BASELINE_TIME_MS / best_time if best_time > 0 else 0,
        "total_input_tokens": total_input_tokens,
        "total_output_tokens": total_output_tokens,
        "total_tokens": total_input_tokens + total_output_tokens,
        "per_iter_input_tokens": per_iter_input_tokens,
        "wall_clock_seconds": elapsed,
    }

In [0]:

class CodeState:
    """Typed persistent state for code optimization."""
    def __init__(self):
        self.history = []
        self.best_time_ms = float("inf")
        self.best_code = ""
        self.best_iteration = -1
        self.iteration = 0
        self.strategy_notes = []
        self.failed_approaches = []

    def add_result(self, code, result):
        speedup = BASELINE_TIME_MS / result["time_ms"] if result["time_ms"] > 0 and result["time_ms"] != float("inf") else 0
        self.history.append({
            "iteration": self.iteration,
            "time_ms": result["time_ms"], "speedup": speedup,
            "correct": result.get("correct", False),
        })
        if result.get("correct", False) and result["time_ms"] < self.best_time_ms:
            self.best_time_ms = result["time_ms"]
            self.best_code = code
            self.best_iteration = self.iteration

    def summary(self) -> str:
        best_speedup = BASELINE_TIME_MS / self.best_time_ms if self.best_time_ms > 0 and self.best_time_ms != float("inf") else 0
        lines = [
            f"Iteration: {self.iteration}/{MAX_ITERATIONS}",
            f"Baseline: {BASELINE_TIME_MS:.3f} ms",
            f"Best: {self.best_time_ms:.3f} ms ({best_speedup:.2f}x, iter {self.best_iteration})",
        ]
        recent = self.history[-5:] if len(self.history) > 5 else self.history
        lines.append("Recent:")
        for h in recent:
            marker = " *BEST*" if h["time_ms"] == self.best_time_ms else ""
            status = "OK" if h["correct"] else "FAIL"
            lines.append(f"  Iter {h['iteration']}: {h['time_ms']:.3f}ms ({h['speedup']:.2f}x) [{status}]{marker}")
        if self.strategy_notes:
            lines.append(f"Strategies: {'; '.join(self.strategy_notes[-3:])}")
        if self.failed_approaches:
            lines.append(f"Failed: {'; '.join(self.failed_approaches[-3:])}")
        return "\n".join(lines)


def run_stateful(run_seed: int = SEED) -> dict:
    """Stateful code optimization. Persistent state + conversation window."""
    optimizer = ChatDatabricks(endpoint=MODEL_ENDPOINT, max_tokens=2048)

    state = CodeState()
    baseline = benchmark_code(DEFAULT_CODE)
    state.add_result(DEFAULT_CODE, baseline)
    current_code = DEFAULT_CODE

    total_input_tokens = 0
    total_output_tokens = 0
    per_iter_input_tokens = []
    messages = []

    start_time = time.time()

    for iteration in range(1, MAX_ITERATIONS + 1):
        state.iteration = iteration

        user_msg = (
            f"{state.summary()}\n\n"
            f"Current best code:\n{state.best_code}\n\n"
            f"Latest benchmark:\n{format_benchmark_result(benchmark_code(current_code), current_code)}\n\n"
            f"Propose an optimized version that runs faster while producing correct output."
        )

        messages.append(HumanMessage(content=user_msg))

        all_msgs = [SystemMessage(content=OPTIMIZER_SYSTEM_PROMPT)] + messages
        inp_tok = sum(count_tokens(m.content) for m in all_msgs)
        total_input_tokens += inp_tok
        per_iter_input_tokens.append(inp_tok)

        response = optimizer.invoke(all_msgs)
        total_output_tokens += count_tokens(response.content)
        messages.append(AIMessage(content=response.content))

        strategy = _extract_strategy(response.content)
        if strategy:
            state.strategy_notes.append(strategy)

        try:
            new_code = _extract_code(response.content)
        except Exception:
            new_code = DEFAULT_CODE

        result = benchmark_code(new_code)

        if not result.get("correct", False):
            state.failed_approaches.append(strategy or f"iter {iteration}")

        state.add_result(new_code, result)
        current_code = new_code if result.get("correct", False) else state.best_code

        speedup = BASELINE_TIME_MS / result["time_ms"] if result["time_ms"] > 0 and result["time_ms"] != float("inf") else 0
        if result.get("correct", False):
            result_msg = (
                f"Result: {result['time_ms']:.3f}ms ({speedup:.2f}x). "
                f"{'NEW BEST!' if result['time_ms'] <= state.best_time_ms else f'Best remains {state.best_time_ms:.3f}ms.'}"
            )
        else:
            result_msg = f"FAILED: {result.get('error', 'unknown')}. Reverting to best."
        messages.append(HumanMessage(content=result_msg))

        if len(messages) > 20:
            messages = messages[-20:]

        print(f"  Iter {iteration}: {result['time_ms']:.3f}ms ({speedup:.2f}x) "
              f"correct={result.get('correct',False)} (best={state.best_time_ms:.3f}ms, inp_tok={inp_tok})")

    elapsed = time.time() - start_time

    return {
        "method": "stateful", "task": "code_opt", "run_seed": run_seed,
        "history": [{"iteration": h["iteration"], "time_ms": h["time_ms"],
                      "speedup": h["speedup"], "correct": h["correct"]} for h in state.history],
        "best_time_ms": state.best_time_ms,
        "best_speedup": BASELINE_TIME_MS / state.best_time_ms if state.best_time_ms > 0 else 0,
        "total_input_tokens": total_input_tokens,
        "total_output_tokens": total_output_tokens,
        "total_tokens": total_input_tokens + total_output_tokens,
        "per_iter_input_tokens": per_iter_input_tokens,
        "wall_clock_seconds": elapsed,
        "strategy_notes": state.strategy_notes,
    }

In [0]:

def _extract_code(text: str) -> str:
    if "CODE:" in text:
        code = text.split("CODE:", 1)[1].strip()
    elif "CODE :" in text:
        code = text.split("CODE :", 1)[1].strip()
    else:
        code = text
    if "```" in code:
        parts = code.split("```")
        if len(parts) >= 2:
            code = parts[1]
            if code.startswith("python"):
                code = code[6:]
            code = code.strip()
    lines = code.split("\n")
    start = 0
    for i, line in enumerate(lines):
        if line.strip().startswith(("def ", "import ", "from ")):
            start = i
            break
    return "\n".join(lines[start:]).strip()

def _extract_strategy(text: str) -> str:
    for line in text.split("\n"):
        if line.strip().upper().startswith("STRATEGY:"):
            return line.strip()[9:].strip()
    return ""

In [0]:

all_results = []

print("=" * 60)
print("CODE OPTIMIZATION BENCHMARK: STATEFUL vs STATELESS")
print("=" * 60)
print(f"Iterations per run: {MAX_ITERATIONS}")
print(f"Runs per condition: {NUM_RUNS}")
print(f"Seeds: {SEEDS}")
print(f"Baseline: {BASELINE_TIME_MS:.3f} ms")
print(f"Model: {MODEL_ENDPOINT}")
print()

for seed in SEEDS[:NUM_RUNS]:
    print(f"\n{'='*60}")
    print(f"SEED {seed}")
    print(f"{'='*60}")

    print(f"\n--- Stateless (seed={seed}) ---")
    t0 = time.time()
    stateless_result = run_stateless(run_seed=seed)
    print(f"  Best: {stateless_result['best_time_ms']:.3f}ms ({stateless_result['best_speedup']:.2f}x)")
    print(f"  Tokens: {stateless_result['total_tokens']:,}")
    print(f"  Time: {time.time()-t0:.1f}s")
    all_results.append(stateless_result)

    print(f"\n--- Stateful (seed={seed}) ---")
    t0 = time.time()
    stateful_result = run_stateful(run_seed=seed)
    print(f"  Best: {stateful_result['best_time_ms']:.3f}ms ({stateful_result['best_speedup']:.2f}x)")
    print(f"  Tokens: {stateful_result['total_tokens']:,}")
    print(f"  Time: {time.time()-t0:.1f}s")
    all_results.append(stateful_result)

In [0]:

print("\n" + "=" * 60)
print("AGGREGATE RESULTS")
print("=" * 60)

for method in ["stateless", "stateful"]:
    runs = [r for r in all_results if r["method"] == method]
    speedups = [r["best_speedup"] for r in runs]
    tokens = [r["total_tokens"] for r in runs]
    inp = [r["total_input_tokens"] for r in runs]
    out = [r["total_output_tokens"] for r in runs]
    times = [r["wall_clock_seconds"] for r in runs]

    print(f"\n{method.upper()}:")
    print(f"  Best speedup:  {np.mean(speedups):.2f}x +/- {np.std(speedups):.2f}")
    print(f"  Total tokens:  {np.mean(tokens):,.0f} +/- {np.std(tokens):,.0f}")
    print(f"  Input tokens:  {np.mean(inp):,.0f} +/- {np.std(inp):,.0f}")
    print(f"  Output tokens: {np.mean(out):,.0f} +/- {np.std(out):,.0f}")
    print(f"  Wall-clock:    {np.mean(times):.1f}s +/- {np.std(times):.1f}s")

In [0]:

import matplotlib
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("Code Optimization: Stateful vs Stateless", fontsize=14, fontweight="bold")

# (a) Per-iteration input token cost
ax = axes[0, 0]
for method, color, label in [("stateless", "#ef4444", "Stateless"), ("stateful", "#2563eb", "Stateful")]:
    runs = [r for r in all_results if r["method"] == method]
    all_curves = [r["per_iter_input_tokens"] for r in runs]
    min_len = min(len(c) for c in all_curves)
    curves = np.array([c[:min_len] for c in all_curves])
    mean_curve = curves.mean(axis=0)
    ax.plot(range(1, min_len+1), mean_curve, color=color, linewidth=2, marker="o", markersize=4, label=label)
    ax.fill_between(range(1, min_len+1), curves.min(axis=0), curves.max(axis=0), color=color, alpha=0.1)
ax.set_xlabel("Iteration")
ax.set_ylabel("Input Tokens")
ax.set_title("(a) Per-Iteration Input Token Cost")
ax.legend()
ax.grid(True, alpha=0.3)

# (b) Convergence (speedup)
ax = axes[0, 1]
for method, color, label in [("stateless", "#ef4444", "Stateless"), ("stateful", "#2563eb", "Stateful")]:
    runs = [r for r in all_results if r["method"] == method]
    curves = []
    for r in runs:
        best_so_far = []
        running_best = 0
        for h in r["history"]:
            if h["correct"]:
                running_best = max(running_best, h["speedup"])
            best_so_far.append(running_best)
        curves.append(best_so_far)
    min_len = min(len(c) for c in curves)
    curves = np.array([c[:min_len] for c in curves])
    mean_curve = curves.mean(axis=0)
    std_curve = curves.std(axis=0)
    ax.plot(range(min_len), mean_curve, color=color, linewidth=2, marker="o", markersize=4, label=label)
    ax.fill_between(range(min_len), mean_curve - std_curve, mean_curve + std_curve, color=color, alpha=0.15)
ax.set_xlabel("Iteration")
ax.set_ylabel("Best Speedup (x)")
ax.set_title("(b) Convergence")
ax.legend()
ax.grid(True, alpha=0.3)

# (c) Total token consumption
ax = axes[1, 0]
for i, (method, color) in enumerate([("stateless", "#ef4444"), ("stateful", "#2563eb")]):
    runs = [r for r in all_results if r["method"] == method]
    inp = np.mean([r["total_input_tokens"] for r in runs])
    out = np.mean([r["total_output_tokens"] for r in runs])
    ax.bar(i, inp, 0.6, color="#93c5fd", edgecolor="#2563eb", label="Input" if i==0 else "")
    ax.bar(i, out, 0.6, bottom=inp, color="#fca5a5", edgecolor="#ef4444", label="Output" if i==0 else "")
    ax.text(i, inp+out+500, f"{int(inp+out):,}", ha="center", fontsize=10, fontweight="bold")
ax.set_xticks([0, 1])
ax.set_xticklabels(["Stateless", "Stateful"])
ax.set_ylabel("Total Tokens")
ax.set_title("(c) Token Consumption")
ax.legend()
ax.grid(True, alpha=0.3, axis="y")

# (d) Wall-clock
ax = axes[1, 1]
for i, (method, color) in enumerate([("stateless", "#fca5a5"), ("stateful", "#93c5fd")]):
    runs = [r for r in all_results if r["method"] == method]
    t = [r["wall_clock_seconds"] for r in runs]
    ax.bar(i, np.mean(t), 0.6, yerr=np.std(t), capsize=5, color=color, edgecolor=["#ef4444","#2563eb"][i])
    ax.text(i, np.mean(t)+np.std(t)+10, f"{np.mean(t):.0f}s", ha="center", fontsize=10, fontweight="bold")
ax.set_xticks([0, 1])
ax.set_xticklabels(["Stateless", "Stateful"])
ax.set_ylabel("Wall-clock time (s)")
ax.set_title("(d) Wall-Clock Time")
ax.grid(True, alpha=0.3, axis="y")

plt.tight_layout()
plt.show()

In [0]:

results_path = "/tmp/code_opt_benchmark_results.json"
with open(results_path, "w") as f:
    json.dump(all_results, f, indent=2, default=str)
print(f"Raw results saved to {results_path}")

import pandas as pd
rows = []
for r in all_results:
    for h in r["history"]:
        rows.append({
            "method": r["method"], "run_seed": r["run_seed"],
            "iteration": h["iteration"], "time_ms": h["time_ms"],
            "speedup": h["speedup"], "correct": h["correct"],
            "total_tokens": r["total_tokens"],
            "total_input_tokens": r["total_input_tokens"],
            "wall_clock_seconds": r["wall_clock_seconds"],
        })
spark.createDataFrame(pd.DataFrame(rows)).write.mode("overwrite").option(
    "overwriteSchema", "true"
).saveAsTable("gc_prod_sandbox_mlproduct.tmp.code_opt_benchmark_results")
print("Results saved to Delta table")